# CTU-13 M4 — Corrected flow-label parser

The previous run preserved **correct structural results** but failed to recognize the CTU-13 `flow=` prefix. This notebook **reuses the downloaded .binetflow and the existing SQLite edge table**, repairing the source label flags only. If the SQLite database was deleted (Colab restarted), it reconstructs the graph. It will only download the capture again if the source file is missing.

**Frozen M3 rules remain unchanged:** δ_TC=0.05; γ_min=3; k_min=2; top-30; weak <2, inconclusive 2–<5, strong ≥5. `Background`, `To-Botnet`, and `To-Normal` never count as verified normal or attacker sources. An `ABSTAIN` decision is not precision 0. Aggregate-only CSVs are exported.


In [1]:
"""CTU-13 Scenario 1: frozen degree-based GSD-PI audit (not an independent proof of attack selectivity).
Data source: CTU-Malware-Capture-Botnet-42, capture20110810.binetflow.
Outputs: aggregate-only CSV and metadata; no raw IPs or per-entity rows exported.
"""
from pathlib import Path
from collections import Counter
import csv, json, math, sqlite3, subprocess
import numpy as np

URL = ('https://mcfp.felk.cvut.cz/publicDatasets/'
       'CTU-Malware-Capture-Botnet-42/detailed-bidirectional-flow-labels/'
       'capture20110810.binetflow')
WORK = Path('/content/ctu13_m4')
SOURCE = WORK / 'capture20110810.binetflow'
DB = WORK / 'ctu13_edges.sqlite'
OUT = WORK / 'ctu13_m4_aggregate.csv'

# Frozen protocol: do not change based on the CTU-13 results.
DELTA_TC = 0.05
MGR_WEAK = 2.0
MGR_STRONG = 5.0
GAMMA_MIN = 3.0
K_MIN = 2
TOP_K = 30
CHUNK = 40_000


def download():
    WORK.mkdir(parents=True, exist_ok=True)
    if SOURCE.exists() and SOURCE.stat().st_size > 300_000_000:
        print('Using existing CTU-13 file:', round(SOURCE.stat().st_size / 1e6, 1), 'MB')
        return
    print('Downloading CTU-13 Scenario 1 directly into Colab...')
    subprocess.run(['wget', '-q', '--show-progress', '-c', '--tries=5', '--timeout=60',
                    '-O', str(SOURCE), URL], check=True)
    if SOURCE.stat().st_size < 100_000_000:
        raise RuntimeError('Downloaded file looks too small; verify server response, not an HTML error page.')
    print('Download complete:', round(SOURCE.stat().st_size / 1e6, 1), 'MB')


def label_class(label):
    # CTU-13 bidirectional flows use "flow=From-Botnet-V42-..." etc.
    # Normalize the explicit flow= prefix, retaining directional semantics.
    s = str(label or '').strip().lower()
    if s.startswith('flow='):
        s = s.split('=', 1)[1].strip()
    if s.startswith('from-botnet'):
        return 'from_botnet', 1
    if s.startswith('from-normal'):
        return 'from_normal', 2
    if s.startswith('to-botnet'):
        return 'to_botnet', 0
    if s.startswith('to-normal'):
        return 'to_normal', 0
    if 'background' in s:
        return 'background', 0
    return 'other', 0


def construct_graph():
    if DB.exists():
        DB.unlink()  # Rebuild from input for a fully reproducible run.
    con = sqlite3.connect(str(DB))
    con.execute('PRAGMA journal_mode=OFF')
    con.execute('PRAGMA synchronous=OFF')
    con.execute('PRAGMA temp_store=MEMORY')
    con.execute('PRAGMA cache_size=-100000')
    con.execute('CREATE TABLE edges (src TEXT NOT NULL, dst TEXT NOT NULL, PRIMARY KEY(src,dst)) WITHOUT ROWID')
    con.execute('CREATE TABLE source_flags (src TEXT PRIMARY KEY, flags INTEGER NOT NULL) WITHOUT ROWID')
    flow_counts = Counter()
    processed = 0
    skipped = 0

    with SOURCE.open('r', encoding='utf-8-sig', errors='replace', newline='') as f:
        reader = csv.DictReader(f)
        fieldmap = {k.strip().lower(): k for k in (reader.fieldnames or [])}
        required = {'srcaddr', 'dstaddr', 'label'}
        if not required.issubset(fieldmap):
            raise RuntimeError(f'Unexpected header: {reader.fieldnames}. Expected SrcAddr,DstAddr,Label')
        sc, dc, lc = [fieldmap[k] for k in ('srcaddr','dstaddr','label')]
        edge_batch = []
        flag_batch = {}

        def flush():
            con.executemany('INSERT OR IGNORE INTO edges (src,dst) VALUES (?,?)', edge_batch)
            if flag_batch:
                con.executemany('''INSERT INTO source_flags (src,flags) VALUES (?,?)
                    ON CONFLICT(src) DO UPDATE SET flags=(source_flags.flags | excluded.flags)''',
                    list(flag_batch.items()))
            con.commit()
            edge_batch.clear()
            flag_batch.clear()

        for row in reader:
            processed += 1
            src = str(row.get(sc) or '').strip()
            dst = str(row.get(dc) or '').strip()
            if not src or not dst or src.lower() in {'nan','none'} or dst.lower() in {'nan','none'}:
                skipped += 1
                continue
            cat, bit = label_class(row.get(lc, ''))
            flow_counts[cat] += 1
            edge_batch.append((src, dst))
            if bit:
                flag_batch[src] = flag_batch.get(src, 0) | bit
            if len(edge_batch) >= CHUNK:
                flush()
                if processed % 400_000 < CHUNK:
                    print(f'Processed {processed:,} flows')
        if edge_batch:
            flush()
    print('Parsed flows:', processed, '; skipped:', skipped)
    print('Flow-label categories:', dict(flow_counts))
    return con, processed, skipped, dict(flow_counts)



def repair_source_labels_using_existing_edges():
    """Restore source labels without rebuilding the already verified edge table.

    Requires the same capture file and SQLite DB from the previous run.
    Labels never participate in structural threshold selection.
    """
    con = sqlite3.connect(str(DB))
    tables = {v[0] for v in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    if not {'edges', 'source_flags'}.issubset(tables):
        con.close()
        raise RuntimeError('Existing DB lacks edges/source_flags; delete DB and rerun full analysis')
    edge_count = con.execute('SELECT count(*) FROM edges').fetchone()[0]
    if edge_count < 1000:
        con.close()
        raise RuntimeError('Existing graph appears incomplete. Delete DB and rerun full analysis')
    con.execute('DELETE FROM source_flags')
    con.commit()
    flow_counts = Counter()
    processed, skipped = 0, 0
    batch = {}
    with SOURCE.open('r', encoding='utf-8-sig', errors='replace', newline='') as inp:
        reader = csv.DictReader(inp)
        fieldmap = {v.strip().lower(): v for v in (reader.fieldnames or [])}
        if not {'srcaddr','dstaddr','label'}.issubset(fieldmap):
            raise ValueError(f'Missing required columns: {reader.fieldnames}')
        sc,dc,lc = (fieldmap[k] for k in ('srcaddr','dstaddr','label'))
        for row in reader:
            processed += 1
            source = str(row.get(sc) or '').strip()
            dest = str(row.get(dc) or '').strip()
            if not source or not dest or source.lower() in ('nan','none') or dest.lower() in ('nan','none'):
                skipped += 1
                continue
            cat, bit = label_class(row.get(lc,''))
            flow_counts[cat] += 1
            if bit:
                batch[source] = batch.get(source,0) | bit
            if processed % 40000 == 0:
                if batch:
                    con.executemany("""INSERT INTO source_flags (src,flags) VALUES (?,?)
                       ON CONFLICT(src) DO UPDATE SET flags=(source_flags.flags | excluded.flags)""", list(batch.items()))
                    con.commit()
                    batch.clear()
        if batch:
            con.executemany("""INSERT INTO source_flags (src,flags) VALUES (?,?)
                ON CONFLICT(src) DO UPDATE SET flags=(source_flags.flags | excluded.flags)""", list(batch.items()))
            con.commit()
    if flow_counts['from_botnet'] == 0 or flow_counts['from_normal'] == 0:
        con.close()
        raise RuntimeError(f'Unexpected label breakdown: {dict(flow_counts)}. Please inspect aggregated label prefixes; no output certified.')
    print('Label audit (aggregate flow counts):',dict(flow_counts))
    print('Existing directed edges retained:',edge_count)
    return con, processed, skipped, dict(flow_counts)


def save_topk_descriptive(con):
    """Post-hoc exploratory Top-k coverage, not an Algorithm 1 deployment outcome.

    No global Precision: most sources have unknown status.
    """
    top = list(con.execute("""SELECT e.src, COUNT(*) AS degree, COALESCE(f.flags,0)
        FROM edges e LEFT JOIN source_flags f ON e.src=f.src
        GROUP BY e.src ORDER BY degree DESC, e.src ASC LIMIT 50"""))
    rows=[]
    for k in (5,10,20,50):
        sub=top[:k]
        pos=sum((flag & 1)!=0 for _,_,flag in sub)
        normal=sum(((flag & 2)!=0 and (flag & 1)==0) for _,_,flag in sub)
        rows.append({'k':k, 'confirmed_botnet_sources':pos,
                     'confirmed_normal_sources':normal,
                     'unknown_source_status':len(sub)-pos-normal,
                     'degree_at_rank_k':sub[-1][1] if sub else None,
                     'interpretation':'descriptive_topk_only_not_precision'})
    path=WORK/'ctu13_m4_topk_known_labels.csv'
    with path.open('w',encoding='utf-8',newline='') as fh:
        w=csv.DictWriter(fh,fieldnames=list(rows[0]))
        w.writeheader();w.writerows(rows)
    print('Saved aggregate Top-k diagnostic:',path)
    return path

def calculate(con, processed, skipped, flow_counts):
    degrees = []
    flags = []
    for d, flag in con.execute('''SELECT COUNT(*) AS deg, COALESCE(f.flags,0)
        FROM edges e LEFT JOIN source_flags f ON e.src=f.src GROUP BY e.src'''):
        degrees.append(d)
        flags.append(flag)
    d = np.asarray(degrees, dtype=np.int64)
    f = np.asarray(flags, dtype=np.int64)
    if len(d) == 0:
        raise RuntimeError('No source entities found')
    x = np.sort(d.astype(np.float64))
    n = len(x)
    gini = float(2.0*np.dot(np.arange(1,n+1),x)/(n*x.sum())-(n+1)/n)
    cv = float(np.std(x)/np.mean(x))
    p90,p95,p97 = [float(v) for v in np.percentile(d, [90,95,97], method='linear')]
    median = float(np.median(d))
    tc = bool(abs(p97-p90)/max(p97,1.0) < DELTA_TC)
    top = np.sort(d[d>1])[::-1][:TOP_K]
    ratios = top[:-1]/top[1:] if len(top)>1 else np.asarray([],float)
    mgr = float(ratios.max()) if len(ratios) else math.nan
    gaps = [(i+1, float(r), int(top[i])) for i,r in enumerate(ratios)
            if r>GAMMA_MIN and i+1>=K_MIN]
    # M3: structural isolation is descriptive; NEVER a prediction of attack selectivity.
    if not np.isfinite(mgr):
        isolation = 'INSUFFICIENT_TAIL_DATA'
    elif mgr < MGR_WEAK:
        isolation = 'WEAK_EXTREME_TAIL_ISOLATION'
    elif mgr < MGR_STRONG:
        isolation = 'INCONCLUSIVE_EXTREME_TAIL_ISOLATION'
    else:
        isolation = 'STRONG_EXTREME_TAIL_CANDIDATE'
    if not tc:
        decision, threshold = 'P95_STABLE', p95
    elif np.isfinite(mgr) and mgr>=MGR_STRONG and gaps:
        decision, threshold = 'GAP_RECOVERY_CANDIDATE', float(gaps[0][2])
    else:
        decision, threshold = 'ABSTAIN_UNRESOLVED_COLLAPSE', None
    chosen = d>=threshold if threshold is not None else np.zeros(n,dtype=bool)
    # From-Botnet identifies source-originating bot traffic, unlike To-Botnet.
    # From-Normal identifies known normal sources. All other sources remain UNKNOWN.
    if flow_counts.get('from_botnet',0)==0 or flow_counts.get('from_normal',0)==0:
        raise RuntimeError('CTU-13 label parsing produced no confirmed source classes; aborting.')
    positive = (f&1)!=0
    normal = ((f&2)!=0) & ~positive
    unknown = ~(positive|normal)
    p_tot, neg_tot = int(positive.sum()), int(normal.sum())
    p_sel, neg_sel = int((positive&chosen).sum()), int((normal&chosen).sum())
    result = {
        'dataset':'CTU-13 Scenario 1 (Neris)',
        'flow_records':processed, 'skipped_flow_records':skipped,
        'active_source_entities':n,
        'unique_source_destination_edges':int(d.sum()),
        'gini_active_sources':gini, 'cv_active_sources':cv,
        'p90_active':p90, 'p95_active':p95, 'p97_active':p97,
        'median_active':median,
        'p95_median_active':(p95/median if median>0 else None),
        'TC':tc, 'MGR_top30_non_singleton':mgr,
        'valid_gap_ranks_gamma3_kmin2':';'.join(str(a[0]) for a in gaps),
        'isolation_diagnostic':isolation,
        'selection_decision':decision, 'threshold':threshold,
        'selected_source_entities':int(chosen.sum()),
        'tail_density_active':float(chosen.mean()) if threshold is not None else None,
        'known_from_botnet_sources':p_tot,
        'known_from_normal_sources':neg_tot,
        'unknown_source_entities':int(unknown.sum()),
        'selected_known_from_botnet':p_sel,
        'selected_known_from_normal':neg_sel,
        'selected_unknown':int((chosen&unknown).sum()),
        'known_botnet_source_capture':(p_sel/p_tot if p_tot else None),
        'known_normal_source_flag_rate':(neg_sel/neg_tot if neg_tot else None),
        'precision':'NOT_IDENTIFIABLE_FROM_PARTIAL_SOURCE_LABELS',
        'lift':'NOT_IDENTIFIABLE_FROM_PARTIAL_SOURCE_LABELS',
        'independent_validation_status':'EXPLORATORY_SCENARIO_ONLY_NOT_FULL_M4',
    }
    for cat,count in sorted(flow_counts.items()):
        result['flows_'+cat] = count
    with OUT.open('w',encoding='utf-8',newline='') as fh:
        writer=csv.DictWriter(fh,fieldnames=list(result))
        writer.writeheader(); writer.writerow(result)
    meta = {k:v for k,v in result.items() if k not in ('dataset',)}
    (WORK/'ctu13_m4_metadata.json').write_text(json.dumps(meta,indent=2,default=str),encoding='utf-8')
    for key in ['active_source_entities','gini_active_sources','p95_active','TC','MGR_top30_non_singleton',
                'isolation_diagnostic','selection_decision','selected_source_entities',
                'known_from_botnet_sources','selected_known_from_botnet','known_from_normal_sources','selected_unknown']:
        print(f'{key}: {result[key]}')
    print('Aggregate-only CSV:', OUT)
    return result


def main():
    download()
    if DB.exists():
        print('Fast correction: retaining original directed edges; repairing labels only.')
        con, processed, skipped, flow_counts = repair_source_labels_using_existing_edges()
    else:
        print('No existing graph database: rebuilding graph with corrected label parser.')
        con, processed, skipped, flow_counts = construct_graph()
    try:
        result=calculate(con, processed, skipped, flow_counts)
        save_topk_descriptive(con)
        return result
    finally:
        con.close()

if __name__ == '__main__':
    result=main()
    # Optional: store ONLY aggregate results permanently in Google Drive.
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        destination=Path('/content/drive/MyDrive/paper2_gsd_results/ctu13_m4')
        destination.mkdir(parents=True,exist_ok=True)
        import shutil
        shutil.copy2(OUT, destination/OUT.name)
        shutil.copy2(WORK/'ctu13_m4_topk_known_labels.csv', destination/'ctu13_m4_topk_known_labels.csv')
        print('Saved aggregate outputs to Drive:', destination)
    except ImportError:
        pass


Download complete: 386.6 MB
No existing graph database: rebuilding graph with corrected label parser.
Processed 400,000 flows
Processed 800,000 flows
Processed 1,200,000 flows
Processed 1,600,000 flows
Processed 2,000,000 flows
Processed 2,400,000 flows
Processed 2,800,000 flows
Parsed flows: 2824636 ; skipped: 0
Flow-label categories: {'background': 2753288, 'from_normal': 30258, 'to_normal': 120, 'other': 9, 'from_botnet': 40961}
active_source_entities: 542093
gini_active_sources: 0.22277429077963684
p95_active: 1.0
TC: True
MGR_top30_non_singleton: 4.245187436676798
isolation_diagnostic: INCONCLUSIVE_EXTREME_TAIL_ISOLATION
selection_decision: ABSTAIN_UNRESOLVED_COLLAPSE
selected_source_entities: 0
known_from_botnet_sources: 1
selected_known_from_botnet: 0
known_from_normal_sources: 6
selected_unknown: 0
Aggregate-only CSV: /content/ctu13_m4/ctu13_m4_aggregate.csv
Saved aggregate Top-k diagnostic: /content/ctu13_m4/ctu13_m4_topk_known_labels.csv
Mounted at /content/drive
Saved aggreg

## Return only the aggregate outputs
Upload `MyDrive/paper2_gsd_results/ctu13_m4/ctu13_m4_aggregate.csv` and optionally `ctu13_m4_topk_known_labels.csv`. Do not upload raw logs, SQLite or IP addresses.
